In [1]:
from pathlib import Path
import sys

project_root = Path.cwd()
while project_root.name != "genai-testing-project":
    project_root = project_root.parent

day2_root = project_root / "interview-prep" / "day2-ai-testing"
sys.path.append(str(day2_root))

from app.tool_registry import (
    build_default_registry,
    ToolSchemaError,
    ToolDisabledError,
    ToolPermissionError,
    ToolNotFoundError,
    ToolAlreadyExistsError,
)

In [2]:
registry = build_default_registry()

In [4]:
from os import name


tools = registry.list_tools(role="tester")
[name["name"] for name in tools]

['calculator', 'customer_lookup', 'ticket_creation', 'knowledge_search']

In [7]:
result = registry.execute_tool("calculator", {"expression": "15 * 7"}, role="tester")

In [9]:
result


{'success': True,
 'tool_name': 'calculator',
 'data': {'expression': '15 * 7', 'result': 105},
 'error': None}

In [11]:
assert result["success"] is True
assert result["data"]["result"] == 105

In [12]:
try:
    registry.execute_tool("calculator", {}, role="tester")
except ToolSchemaError as error:
    print("Schema validation failed as expected")
    print(error)

Schema validation failed as expected
'expression' is a required property

Failed validating 'required' in schema:
    {'type': 'object',
     'properties': {'expression': {'type': 'string',
                                   'description': 'Arithmetic expression, '
                                                  "for example '15 * 7'."}},
     'required': ['expression'],
     'additionalProperties': False}

On instance:
    {}


In [13]:
registry = build_default_registry()
registry.disable_tool("calculator")

try:
    registry.execute_tool(
        "calculator",
        {"expression": "15 * 7"},
        role="tester",
    )
except ToolDisabledError as error:
    print("Disabled tool blocked as expected")
    print(error)

Disabled tool blocked as expected
Tool is disabled: calculator


In [14]:
registry = build_default_registry()

try:
    registry.execute_tool(
        "calculator",
        {"expression": "15 * 7"},
        role="viewer",
    )
except ToolPermissionError as error:
    print("Permission blocked as expected")
    print(error)

Permission blocked as expected
Role 'viewer' cannot execute tool 'calculator'


In [15]:
registry = build_default_registry()

try:
    registry.execute_tool(
        "delete_all_users",
        {},
        role="admin",
    )
except ToolNotFoundError as error:
    print("Unknown tool blocked as expected")
    print(error)

Unknown tool blocked as expected
Tool not registered: delete_all_users


In [16]:
from app.mcp_schemas import MCP_TOOL_SCHEMAS
from app.tools import TOOL_FUNCTIONS

registry = build_default_registry()

try:
    registry.register_tool(
        MCP_TOOL_SCHEMAS["calculator"],
        TOOL_FUNCTIONS["calculator"],
    )
except ToolAlreadyExistsError as error:
    print("Duplicate tool blocked as expected")
    print(error)

Duplicate tool blocked as expected
Tool already registered: calculator
